# FathomNet 2026 — Exploratory Data Analysis

This notebook explores `dataset_train.json` (and optionally `dataset_test.json`) to understand:
- Class distribution
- Annotation density per image
- Sample images with bounding boxes
- Image size distribution
- Unlabeled image analysis (key for the PU learning setup)

**Run from the project root** so that relative paths resolve correctly.

In [ ]:
import json
import os
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# Reproducibility
random.seed(42)
np.random.seed(42)

# Adjust these if you run from a different working directory
TRAIN_ANN = "data/annotations/dataset_train.json"
TEST_ANN  = "data/annotations/dataset_test.json"
IMG_DIR   = "data/raw"

print("Loading annotations...")
with open(TRAIN_ANN) as f:
    train_coco = json.load(f)
with open(TEST_ANN) as f:
    test_coco = json.load(f)
print(f"Train images: {len(train_coco['images'])}, annotations: {len(train_coco['annotations'])}")
print(f"Test  images: {len(test_coco['images'])}, annotations: {len(test_coco['annotations'])}")

In [ ]:
# ── Build helper data structures ──────────────────────────────────────────────
categories = {c['id']: c['name'] for c in train_coco['categories']}
cat_names  = [c['name'] for c in sorted(train_coco['categories'], key=lambda c: c['id'])]

# Annotation counts per image
ann_per_image: dict[int, list] = defaultdict(list)
for ann in train_coco['annotations']:
    ann_per_image[ann['image_id']].append(ann)

# Images with zero annotations = potentially unlabeled positives (key PU insight)
all_image_ids = {img['id'] for img in train_coco['images']}
labeled_ids   = set(ann_per_image.keys())
unlabeled_ids = all_image_ids - labeled_ids
print(f"Images with ≥1 annotation : {len(labeled_ids)}")
print(f"Images with 0 annotations : {len(unlabeled_ids)} (unlabeled positives in PU setting)")

## 1. Class Distribution

In [ ]:
# Count annotations per category
cat_counts = Counter(ann['category_id'] for ann in train_coco['annotations'])
df_cats = pd.DataFrame([
    {'category': categories[cid], 'count': cnt}
    for cid, cnt in cat_counts.most_common()
])

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(df_cats['category'], df_cats['count'], color='steelblue')
ax.set_xlabel('Number of annotations')
ax.set_title('Class distribution — training set')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/logs/class_distribution.png', dpi=150)
plt.show()

print(df_cats.to_string(index=False))

## 2. Annotation Density per Image

In [ ]:
# Distribution of annotation count per image (among labeled images)
ann_counts = [len(anns) for anns in ann_per_image.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(ann_counts, bins=40, color='coral', edgecolor='white')
axes[0].set_xlabel('Annotations per image')
axes[0].set_ylabel('Count')
axes[0].set_title('Annotation density (labeled images only)')

# Log scale to see the long tail
axes[1].hist(ann_counts, bins=40, color='coral', edgecolor='white', log=True)
axes[1].set_xlabel('Annotations per image')
axes[1].set_ylabel('Count (log)')
axes[1].set_title('Annotation density (log scale)')

plt.tight_layout()
plt.savefig('outputs/logs/annotation_density.png', dpi=150)
plt.show()

print(f"Median annotations/image: {np.median(ann_counts):.1f}")
print(f"Mean   annotations/image: {np.mean(ann_counts):.1f}")
print(f"Max    annotations/image: {max(ann_counts)}")

## 3. Image Size Distribution

In [ ]:
widths  = [img['width']  for img in train_coco['images']]
heights = [img['height'] for img in train_coco['images']]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths,  bins=30, color='seagreen')
axes[0].set_title('Image widths')
axes[0].set_xlabel('Pixels')

axes[1].hist(heights, bins=30, color='seagreen')
axes[1].set_title('Image heights')
axes[1].set_xlabel('Pixels')

axes[2].scatter(widths, heights, alpha=0.3, s=5, color='seagreen')
axes[2].set_title('Width vs Height')
axes[2].set_xlabel('Width')
axes[2].set_ylabel('Height')

plt.tight_layout()
plt.savefig('outputs/logs/image_sizes.png', dpi=150)
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

## 4. Bounding Box Size Distribution

In [ ]:
# Relative bbox sizes help decide the YOLO input resolution
img_info = {img['id']: img for img in train_coco['images']}

rel_widths, rel_heights, rel_areas = [], [], []
for ann in train_coco['annotations']:
    img = img_info[ann['image_id']]
    x, y, w, h = ann['bbox']
    rel_widths.append(w / img['width'])
    rel_heights.append(h / img['height'])
    rel_areas.append((w * h) / (img['width'] * img['height']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title in zip(axes,
                            [rel_widths, rel_heights, rel_areas],
                            ['Relative bbox width', 'Relative bbox height', 'Relative bbox area']):
    ax.hist(data, bins=40, color='mediumpurple')
    ax.set_title(title)
    ax.set_xlabel('Fraction of image dimension')

plt.tight_layout()
plt.savefig('outputs/logs/bbox_sizes.png', dpi=150)
plt.show()

print(f"Median relative bbox area: {np.median(rel_areas):.4f}")
small = sum(1 for a in rel_areas if a < 0.01) / len(rel_areas)
print(f"Fraction of 'small' objects (area < 1% of image): {small:.1%}")

## 5. Sample Images with Bounding Boxes

In [ ]:
def load_image(img_info: dict, img_dir: str) -> Image.Image | None:
    """Try to load an image from disk; return None if not found."""
    for subdir in ['train', 'test', '']:
        p = Path(img_dir) / subdir / img_info['file_name']
        if p.exists():
            return Image.open(p).convert('RGB')
    return None


def show_sample_images(image_ids, img_info, ann_by_img, categories, img_dir,
                       title='Samples', n_cols=4, max_n=8):
    image_ids = [iid for iid in image_ids if load_image(img_info[iid], img_dir) is not None]
    sample_ids = random.sample(image_ids, min(max_n, len(image_ids)))
    n_cols = min(n_cols, len(sample_ids))
    n_rows = (len(sample_ids) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    cmap = plt.get_cmap('tab20')
    cat_ids = sorted(categories.keys())
    color_map = {cid: cmap(i / len(cat_ids)) for i, cid in enumerate(cat_ids)}

    for ax, img_id in zip(axes, sample_ids):
        info = img_info[img_id]
        img  = load_image(info, img_dir)
        ax.imshow(img)

        for ann in ann_by_img.get(img_id, []):
            x, y, w, h = ann['bbox']
            cid = ann['category_id']
            color = color_map[cid]
            rect = patches.Rectangle((x, y), w, h,
                                      linewidth=1.5, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y - 4, categories[cid], color=color, fontsize=7,
                    bbox=dict(facecolor='black', alpha=0.4, pad=1, linewidth=0))

        ax.set_title(f"id={img_id} | {len(ann_by_img.get(img_id, []))} anns", fontsize=8)
        ax.axis('off')

    for ax in axes[len(sample_ids):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# Labeled samples
show_sample_images(
    list(labeled_ids), img_info, ann_per_image, categories, IMG_DIR,
    title='Labeled images with bounding boxes'
)

In [ ]:
# Unlabeled samples — these are the images the PU model needs to handle
if unlabeled_ids:
    show_sample_images(
        list(unlabeled_ids), img_info, ann_per_image, categories, IMG_DIR,
        title='Unlabeled images (0 annotations in training set)',
        max_n=8
    )
else:
    print("No zero-annotation images found — all images have at least one annotation.")

## 6. Per-Category Image Coverage

How many *unique images* contain each category? A class that appears in very few images will be harder to detect.

In [ ]:
imgs_per_cat: dict[int, set] = defaultdict(set)
for ann in train_coco['annotations']:
    imgs_per_cat[ann['category_id']].add(ann['image_id'])

df_coverage = pd.DataFrame([
    {'category': categories[cid], 'unique_images': len(imgs)}
    for cid, imgs in imgs_per_cat.items()
]).sort_values('unique_images', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(df_coverage['category'], df_coverage['unique_images'], color='darkorange')
ax.set_xlabel('Number of unique images')
ax.set_title('Per-category image coverage')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('outputs/logs/category_coverage.png', dpi=150)
plt.show()

## 7. Summary Statistics

In [ ]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Training images total      : {len(train_coco['images'])}")
print(f"  Labeled (≥1 annotation)  : {len(labeled_ids)}")
print(f"  Unlabeled (0 annotations): {len(unlabeled_ids)}")
print(f"Training annotations total : {len(train_coco['annotations'])}")
print(f"Test images                : {len(test_coco['images'])}")
print(f"Categories                 : {len(categories)}")
print(f"")
print(f"Avg annotations / labeled image: {np.mean(ann_counts):.1f}")
print(f"Median rel. bbox area          : {np.median(rel_areas):.4f}")
print(f"Small objects (area < 1%)      : {small:.1%}")